### Feature Engineering for Credit Risk Modeling

#### Objective
The goal of this notebook is to transform raw applicant data into model-ready features
based on insights from EDA.

Key principles:
- Preserve monotonic relationships
- Handle skewed distributions
- Encode categorical variables appropriately
- Avoid data leakage


In [14]:
import numpy as np
import pandas as pd

In [15]:
df = pd.read_csv('C:/Users/Pratik/DS/credit-risk-ml/data/raw/application_train.csv')

In [16]:
# Numeric Feature Transformations
df['AMT_INCOME_LOG'] = np.log1p(df['AMT_INCOME_TOTAL'])
df['AMT_CREDIT_LOG'] = np.log1p(df['AMT_CREDIT'])
df['AMT_ANNUITY_LOG'] = np.log1p(df['AMT_ANNUITY'])
df['AMT_GOODS_LOG'] = np.log1p(df['AMT_GOODS_PRICE'])

In [17]:
df['AGE_YEARS'] = - df['DAYS_BIRTH'] / 365
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['EMPLOYED_YEARS'] = df['DAYS_EMPLOYED'].where(df['DAYS_EMPLOYED'].notna(),np.nan) / 365 
df.drop(columns=['DAYS_EMPLOYED','DAYS_BIRTH'],inplace=True)

In [18]:
# Creating Ratio Features
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / df['AMT_CREDIT']
df['CHILDREN_RATIO'] = df['CNT_CHILDREN'] / df['CNT_FAM_MEMBERS']

In [19]:
# Handling numerical missing values
num_cols = df.select_dtypes(include='number')
df[num_cols.columns] = num_cols.fillna(num_cols.median())

In [20]:
# Binary Features
df['HAS_CHILDREN'] = (df['CNT_CHILDREN'] > 0).astype('Int64')
df['IS_SINGLE'] = (df['CNT_FAM_MEMBERS'] == 1).astype('Int64')
df['HAS_CAR'] = (df['FLAG_OWN_CAR'] == 'Y').astype('Int64')
df['HAS_REALTY'] = (df['FLAG_OWN_REALTY'] == 'Y').astype('Int64')
df['LONG_EMPLOYED'] = np.where(
    df['EMPLOYED_YEARS'].isna(),
    np.nan,
    (df['EMPLOYED_YEARS'] > 5).astype(int)
)
df.drop(columns=['FLAG_OWN_CAR', 'FLAG_OWN_REALTY'], inplace=True)

C:\Users\Pratik\AppData\Local\Temp\ipykernel_18460\4243978639.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_CHILDREN'] = (df['CNT_CHILDREN'] > 0).astype('Int64')
C:\Users\Pratik\AppData\Local\Temp\ipykernel_18460\4243978639.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['IS_SINGLE'] = (df['CNT_FAM_MEMBERS'] == 1).astype('Int64')
C:\Users\Pratik\AppData\Local\Temp\ipykernel_18460\4243978639.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti

Handling binary features with missing values
Binary features derived from raw numeric variables preserve missing values where NaN represents “unknown” rather than “false”. This prevents conflating missing employment history with short employment duration, which is especially important in credit risk modeling.

In [21]:
# Ordinal Category Encoding
# Encoding Education level 
education_map = {
    'Lower secondary': 0,
    'Secondary / secondary special': 1,
    'Incomplete higher': 2,
    'Higher education': 3,
    'Academic degree': 4
}

df['NAME_EDUCATION_ENC'] = df['NAME_EDUCATION_TYPE'].map(education_map)

C:\Users\Pratik\AppData\Local\Temp\ipykernel_18460\1614994837.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['NAME_EDUCATION_ENC'] = df['NAME_EDUCATION_TYPE'].map(education_map)


In [22]:
# Grouping Income Types
income_map = {
    "Working": "Employed",
    "Commercial associate": "Employed",
    "State servant": "Employed",

    "Pensioner": "Pension",
    "Student": "Non-working",

    "Unemployed": "Non-working",
    "Maternity leave": "Non-working",

    "Businessman": "Self-employed",

    "Other": "Other"
}

df["INCOME_GROUP"] = df["NAME_INCOME_TYPE"].map(income_map)

C:\Users\Pratik\AppData\Local\Temp\ipykernel_18460\948974872.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["INCOME_GROUP"] = df["NAME_INCOME_TYPE"].map(income_map)


In [23]:
pd.get_dummies(df["INCOME_GROUP"], prefix="INC", drop_first=True)

,INC_Non-working,INC_Pension,INC_Self-employed
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
...,...,...,...
307506,False,False,False
307507,False,True,False
307508,False,False,False
307509,False,False,False


In [ ]:
# Handling Missing Values in Category Columns
cat_col = df.select_dtypes(include='object')
df[cat_col.columns] = df[cat_col.columns].fillna('Unknown')

In [25]:
df.isnull().sum()

SK_ID_CURR            0
TARGET                0
NAME_CONTRACT_TYPE    0
CODE_GENDER           0
CNT_CHILDREN          0
                     ..
HAS_CAR               0
HAS_REALTY            0
LONG_EMPLOYED         0
NAME_EDUCATION_ENC    0
INCOME_GROUP          0
Length: 135, dtype: int64

In [26]:
df.to_csv("C:/Users/Pratik/DS/credit-risk-ml/data/processed/credit_features_v1.csv", index=False)